# Pandas y SQL: tablas, llaves y el stack columnar moderno

**Ciencia de Datos, Sección A** · Sesión 3 · 28 de julio de 2026

Notebook companion de la presentación. Corran cada celda con `Shift+Enter`.

Dependencias: `pip install pandas duckdb pyarrow polars lxml`

## 1. Del array al DataFrame

Un DataFrame es un diccionario ordenado de columnas, y cada columna es (casi siempre) un array de NumPy con una etiqueta encima.

In [1]:
import numpy as np
import pandas as pd

s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s.values)   # array de NumPy adentro
print(s["b"])     # acceso por etiqueta

[10 20 30]
20


In [2]:
df = pd.DataFrame({
    "producto": ["cafe", "azucar", "harina"],
    "precio": np.array([28.5, 9.0, 12.75]),
    "kilos": np.array([120, 340, 260]),
})
print(df)
print(df.dtypes)   # cada columna con su propio dtype

  producto  precio  kilos
0     cafe   28.50    120
1   azucar    9.00    340
2   harina   12.75    260
producto     object
precio      float64
kilos         int64
dtype: object


## 2. Explorar y seleccionar

`tips`: propinas de un restaurante, el dataset clásico de seaborn.

In [3]:
URL = ("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")

tips = pd.read_csv(URL)
print(tips.shape)
tips.head()

(244, 7)


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [4]:
tips.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   total_bill  244 non-null    float64
 1   tip         244 non-null    float64
 2   sex         244 non-null    object 
 3   smoker      244 non-null    object 
 4   day         244 non-null    object 
 5   time        244 non-null    object 
 6   size        244 non-null    int64  
dtypes: float64(2), int64(1), object(4)
memory usage: 13.5+ KB


In [5]:
print(tips.describe())
print(tips["day"].value_counts())
print(tips.isna().sum())

       total_bill         tip        size
count  244.000000  244.000000  244.000000
mean    19.785943    2.998279    2.569672
std      8.902412    1.383638    0.951100
min      3.070000    1.000000    1.000000
25%     13.347500    2.000000    2.000000
50%     17.795000    2.900000    2.000000
75%     24.127500    3.562500    3.000000
max     50.810000   10.000000    6.000000
day
Sat     87
Sun     76
Thur    62
Fri     19
Name: count, dtype: int64
total_bill    0
tip           0
sex           0
smoker        0
day           0
time          0
size          0
dtype: int64


In [6]:
print(tips["tip"].head())                 # una Series
print(tips[["tip", "total_bill"]].head())  # un DataFrame
print(tips.loc[0:4, ["day", "tip"]])       # por ETIQUETA
print(tips.iloc[0:5, 0:2])                 # por POSICION

0    1.01
1    1.66
2    3.50
3    3.31
4    3.61
Name: tip, dtype: float64
    tip  total_bill
0  1.01       16.99
1  1.66       10.34
2  3.50       21.01
3  3.31       23.68
4  3.61       24.59
   day   tip
0  Sun  1.01
1  Sun  1.66
2  Sun  3.50
3  Sun  3.31
4  Sun  3.61
   total_bill   tip
0       16.99  1.01
1       10.34  1.66
2       21.01  3.50
3       23.68  3.31
4       24.59  3.61


In [7]:
print(tips[tips["total_bill"] > 30].shape)
print(tips[(tips["day"] == "Sun") & (tips["size"] >= 4)].shape)

tips["pct"] = tips["tip"] / tips["total_bill"]
tips[["total_bill", "tip", "pct"]].head()

(32, 7)
(22, 7)


,total_bill,tip,pct
0,16.99,1.01,0.059447
1,10.34,1.66,0.160542
2,21.01,3.50,0.166587
3,23.68,3.31,0.139780
4,24.59,3.61,0.146808


## 3. Agregar y combinar

`groupby` es split (partir por categoría), apply (calcular) y combine (juntar). `agg` resume; `transform` pega el valor del grupo a cada fila original.

In [8]:
print(tips.groupby("day")["tip"].mean())
print(tips.groupby(["day", "time"])["total_bill"].sum())

day
Fri     2.734737
Sat     2.993103
Sun     3.255132
Thur    2.771452
Name: tip, dtype: float64
day   time  
Fri   Dinner     235.96
      Lunch       89.92
Sat   Dinner    1778.40
Sun   Dinner    1627.16
Thur  Dinner      18.78
      Lunch     1077.55
Name: total_bill, dtype: float64


In [9]:
tips.groupby("day").agg(
    propina_media=("tip", "mean"),
    cuenta_max=("total_bill", "max"),
    n=("tip", "size"),
)

,propina_media,cuenta_max,n
day,,,
Fri,2.734737,40.17,19
Sat,2.993103,50.81,87
Sun,3.255132,48.17,76
Thur,2.771452,43.11,62


In [25]:
tips["media_dia"] = tips.groupby("day")["pct"].transform("mean")
tips[["day", "pct", "media_dia"]].head(100)

,day,pct,media_dia
0,Sun,0.059447,0.166897
1,Sun,0.160542,0.166897
2,Sun,0.166587,0.166897
3,Sun,0.139780,0.166897
4,Sun,0.146808,0.166897
...,...,...,...
95,Fri,0.117750,0.169913
96,Fri,0.146628,0.169913
97,Fri,0.124688,0.169913
98,Fri,0.142789,0.169913


In [11]:
ventas = pd.DataFrame({
    "sucursal": [1, 2, 1, 3],
    "monto": [250, 480, 120, 310]})
sucursales = pd.DataFrame({
    "sucursal": [1, 2, 4],
    "ciudad": ["Guatemala", "Xela", "Peten"]})

print(pd.merge(ventas, sucursales, on="sucursal", how="inner"))
print(pd.merge(ventas, sucursales, on="sucursal", how="left"))

   sucursal  monto     ciudad
0         1    250  Guatemala
1         2    480       Xela
2         1    120  Guatemala
   sucursal  monto     ciudad
0         1    250  Guatemala
1         2    480       Xela
2         1    120  Guatemala
3         3    310        NaN


## 4. SQL

`merge` y `JOIN` son la misma operación; `groupby` y `GROUP BY` son la misma idea. Usamos DuckDB para correr SQL sobre nuestros DataFrames.

In [12]:
import duckdb

duckdb.sql("""
    SELECT day,
           AVG(tip) AS propina_media,
           COUNT(*) AS n
    FROM tips
    WHERE total_bill > 10
    GROUP BY day
    HAVING COUNT(*) > 20
    ORDER BY propina_media DESC
    LIMIT 3
""").df()

,day,propina_media,n
0,Sun,3.286761,71
1,Sat,3.078434,83
2,Thur,2.900536,56


In [13]:
duckdb.sql("""
    SELECT v.sucursal, s.ciudad, SUM(v.monto) AS total
    FROM ventas AS v
    INNER JOIN sucursales AS s
            ON v.sucursal = s.sucursal
    GROUP BY v.sucursal, s.ciudad
    ORDER BY total DESC
""").df()

,sucursal,ciudad,total
0,2,Xela,480.0
1,1,Guatemala,370.0


## 5. Window functions

Un `GROUP BY` devuelve una fila por grupo. Una window function calcula sobre el grupo pero conserva todas las filas: es el `transform` de pandas, en SQL.

In [14]:
duckdb.sql("""
    SELECT day, total_bill, tip,
           RANK() OVER (PARTITION BY day
                        ORDER BY total_bill DESC) AS pos,
           AVG(tip) OVER (PARTITION BY day) AS media_dia
    FROM tips
    LIMIT 10
""").df()

,day,total_bill,tip,pos,media_dia
0,Sun,48.17,5.00,1,3.255132
1,Sun,45.35,3.50,2,3.255132
2,Sun,40.55,3.00,3,3.255132
3,Sun,38.07,4.00,4,3.255132
4,Sun,35.26,5.00,5,3.255132
5,Sun,34.81,5.20,6,3.255132
6,Sun,34.65,3.68,7,3.255132
7,Sun,34.63,3.55,8,3.255132
8,Sun,32.90,3.11,9,3.255132
9,Sun,32.40,6.00,10,3.255132


In [15]:
duckdb.sql("""
    SELECT day, size, total_bill,
           SUM(total_bill) OVER (
               PARTITION BY day
               ORDER BY size) AS acumulado,
           LAG(total_bill) OVER (
               PARTITION BY day
               ORDER BY size) AS anterior
    FROM tips
    LIMIT 10
""").df()

,day,size,total_bill,acumulado,anterior
0,Sat,1,3.07,10.32,NaN
1,Sat,1,7.25,10.32,3.07
2,Sat,2,15.81,902.69,7.25
3,Sat,2,17.92,902.69,15.81
4,Sat,2,22.67,902.69,17.92
5,Sat,2,19.82,902.69,22.67
6,Sat,2,27.18,902.69,19.82
7,Sat,2,13.37,902.69,27.18
8,Sat,2,12.69,902.69,13.37
9,Sat,2,21.70,902.69,12.69


## 6. DuckDB, Parquet y Polars

DuckDB es una base de datos analítica sin servidor: consulta lo que ya está en memoria y también archivos en disco.

In [16]:
duckdb.sql("""
    SELECT day, AVG(tip) AS media, COUNT(*) AS n
    FROM tips
    GROUP BY day
    ORDER BY media DESC
""").df()

,day,media,n
0,Sun,3.255132,76
1,Sat,2.993103,87
2,Thur,2.771452,62
3,Fri,2.734737,19


In [17]:
tips.to_parquet("tips.parquet")

duckdb.sql("""SELECT day, SUM(tip)
              FROM 'tips.parquet'
              GROUP BY day""").df()

,day,sum(tip)
0,Fri,51.96
1,Sat,260.40
2,Sun,247.39
3,Thur,171.83


In [18]:
import polars as pl

print(pl.read_parquet("tips.parquet").head())

# lazy: no lee nada hasta collect()
(pl.scan_parquet("tips.parquet")
   .filter(pl.col("total_bill") > 20)
   .group_by("day").agg(pl.col("tip").mean())
   .collect())

shape: (5, 9)
┌────────────┬──────┬────────┬────────┬───┬────────┬──────┬──────────┬───────────┐
│ total_bill ┆ tip  ┆ sex    ┆ smoker ┆ … ┆ time   ┆ size ┆ pct      ┆ media_dia │
│ ---        ┆ ---  ┆ ---    ┆ ---    ┆   ┆ ---    ┆ ---  ┆ ---      ┆ ---       │
│ f64        ┆ f64  ┆ str    ┆ str    ┆   ┆ str    ┆ i64  ┆ f64      ┆ f64       │
╞════════════╪══════╪════════╪════════╪═══╪════════╪══════╪══════════╪═══════════╡
│ 16.99      ┆ 1.01 ┆ Female ┆ No     ┆ … ┆ Dinner ┆ 2    ┆ 0.059447 ┆ 0.166897  │
│ 10.34      ┆ 1.66 ┆ Male   ┆ No     ┆ … ┆ Dinner ┆ 3    ┆ 0.160542 ┆ 0.166897  │
│ 21.01      ┆ 3.5  ┆ Male   ┆ No     ┆ … ┆ Dinner ┆ 3    ┆ 0.166587 ┆ 0.166897  │
│ 23.68      ┆ 3.31 ┆ Male   ┆ No     ┆ … ┆ Dinner ┆ 2    ┆ 0.13978  ┆ 0.166897  │
│ 24.59      ┆ 3.61 ┆ Female ┆ No     ┆ … ┆ Dinner ┆ 4    ┆ 0.146808 ┆ 0.166897  │
└────────────┴──────┴────────┴────────┴───┴────────┴──────┴──────────┴───────────┘


day,tip
str,f64
"""Fri""",3.58
"""Sat""",3.842368
"""Sun""",3.98
"""Thur""",4.150625


## 7. Ejercicios

Completen los `# ¿Que va aqui?`.

### Ejercicio 1: propinas por día y turno

In [19]:
# a) porcentaje medio de propina por dia
# b) por dia Y turno (day, time), en una tabla
# c) el dia con el mayor porcentaje medio
# d) numero de mesas por dia

resumen = ...  # ¿Que va aqui? (usen agg)

# Verificacion: resumen debe tener 4 filas
# print(resumen.shape)

### Ejercicio 2: un merge con llave imperfecta

In [20]:
metas = pd.DataFrame({
    "day": ["Thur", "Fri", "Sat"],
    "meta_propina": [0.16, 0.16, 0.18]})

# a) peguen metas a tips con how="left"
# b) ¿que dia quedo sin meta? ¿por que?
# c) columna cumple = pct >= meta_propina
# d) % de mesas que cumplen, por dia

# ¿Que va aqui?

# Verificacion: comparen shape antes y despues
# print(tips.shape)

### Ejercicio 3: la misma pregunta, dos veces

Las 3 cuentas más altas de cada día, con su posición.

In [21]:
# Version A: pandas
# Pista: sort_values + groupby(...).head(3)
top_pandas = ...   # ¿Que va aqui?

# Version B: DuckDB con window function
top_sql = duckdb.sql("""
    SELECT 1  /* ¿Que va aqui? RANK OVER PARTITION */
""").df()

# Verificacion: ambos deben dar 4*3 = 12 filas
# print(len(top_pandas), len(top_sql))